# 01 Demo checkout tren Google Colab

Notebook nay dung de test model da train tren anh khay com moi. No khong train lai model.

Neu co `egg_fish_detector.pt`, demo se dung YOLO lam bang chung phu de dem trung va phat hien ca trong canh chua.


## 1. Clone hoac pull code moi nhat

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Vo-Minh-Tri1412/cnn-food-recognition.git"
BRANCH = "codex/colab-kaggle-workflow"
PROJECT_ROOT = Path("/content/cnn-food-recognition")


def run(cmd, cwd=None, env=None):
    print("+", " ".join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], cwd=cwd, env=env, check=True)

if not PROJECT_ROOT.exists():
    run(["git", "clone", "-b", BRANCH, REPO_URL, PROJECT_ROOT])
else:
    run(["git", "fetch", "origin"], cwd=PROJECT_ROOT)
    run(["git", "checkout", BRANCH], cwd=PROJECT_ROOT)
    run(["git", "pull", "origin", BRANCH], cwd=PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("BRANCH =", BRANCH)


## 2. Cai dependency nhe cho demo

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "opencv-python", "pillow", "pandas", "matplotlib"])


## 3. Mount Google Drive va tim model moi nhat

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/canteen_checkout")
DRIVE_RUNS_DIR = DRIVE_ROOT / "runs"
DRIVE_MODELS_DIR = DRIVE_ROOT / "models"
print("DRIVE_ROOT =", DRIVE_ROOT)

def newest_path(paths):
    paths = [Path(p) for p in paths if Path(p).exists()]
    if not paths:
        return None
    return max(paths, key=lambda p: p.stat().st_mtime)

model_candidates = list(DRIVE_RUNS_DIR.rglob("dish_classifier.pt")) + list(DRIVE_MODELS_DIR.rglob("dish_classifier.pt"))
MODEL_PATH = newest_path(model_candidates)
if MODEL_PATH is None:
    raise FileNotFoundError("Khong tim thay dish_classifier.pt. Hay train notebook 00 truoc hoac copy model vao Drive.")
print("MODEL_PATH =", MODEL_PATH)

detector_candidates = list(DRIVE_RUNS_DIR.rglob("egg_fish_detector.pt")) + list(DRIVE_MODELS_DIR.rglob("egg_fish_detector.pt"))
DETECTOR_PATH = newest_path(detector_candidates)
print("DETECTOR_PATH =", DETECTOR_PATH if DETECTOR_PATH else "khong co, demo se chay classifier-only")

## 4. Upload anh khay com can demo

Chay cell nay roi chon 1 anh khay com tu may tinh.

In [ ]:
from google.colab import files

UPLOAD_DIR = Path("/content/canteen_demo_images")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("Chua upload anh nao.")

first_name = next(iter(uploaded))
DEMO_IMAGE = UPLOAD_DIR / first_name
DEMO_IMAGE.write_bytes(uploaded[first_name])
print("DEMO_IMAGE =", DEMO_IMAGE)


## 5. Tuy chinh demo

- `THRESHOLD`: confidence duoi nguong nay se danh dau uncertain va khong tinh tien neu logic bill yeu cau.
- `IGNORE_REGIONS`: ten o crop muon bo qua, vi du `['center']`.
- `REGIONS_JSON`: file crop region rieng neu ban co, con khong thi dung preset 5 o.

In [ ]:
THRESHOLD = 0.45
EGG_COUNT = 1
USE_DETECTOR = DETECTOR_PATH is not None
DETECTOR_THRESHOLD = 0.25
IGNORE_REGIONS = []
REGIONS_JSON = None

print({
    "threshold": THRESHOLD,
    "egg_count": EGG_COUNT,
    "use_detector": USE_DETECTOR,
    "detector_threshold": DETECTOR_THRESHOLD,
    "ignore_regions": IGNORE_REGIONS,
    "regions_json": REGIONS_JSON,
})

## 6. Chay checkout

In [ ]:
cmd = [
    sys.executable, "scripts/06_demo_checkout.py",
    "--image", DEMO_IMAGE,
    "--model", MODEL_PATH,
    "--threshold", str(THRESHOLD),
    "--egg-count", str(EGG_COUNT),
]
if USE_DETECTOR and DETECTOR_PATH:
    cmd += ["--use-detector", "--detector", DETECTOR_PATH, "--detector-threshold", str(DETECTOR_THRESHOLD)]
if REGIONS_JSON:
    cmd += ["--regions-json", REGIONS_JSON]
for region in IGNORE_REGIONS:
    cmd += ["--ignore-region", region]
run(cmd, cwd=PROJECT_ROOT)

## 7. Hien crop va bill

In [ ]:
from IPython.display import Image as IPImage, display

bill_path = PROJECT_ROOT / "outputs" / "bills" / f"{DEMO_IMAGE.stem}_bill.json"
if not bill_path.exists():
    raise FileNotFoundError(bill_path)

bill = json.loads(bill_path.read_text(encoding="utf-8"))
print(json.dumps(bill, ensure_ascii=False, indent=2))
print("Total VND =", bill["total_vnd"])
print("Detector loaded =", bill.get("detector_loaded"), "| detector =", bill.get("detector_path"))

for idx, item in enumerate(bill["items"], 1):
    raw = item.get("raw_class_name", item.get("class_name"))
    final = item.get("class_name")
    evidence = f"egg={item.get('egg_count', 0)} fish={item.get('fish_count', 0)} reason={item.get('fusion_reason', '')}"
    print(f"\n{idx}. raw={raw} -> final={final} | conf={item['confidence']:.2f} | price={item['price_vnd']:,} VND | {evidence}")
    crop_path = Path(item["crop_path"])
    if crop_path.exists():
        display(IPImage(filename=str(crop_path), width=220))